In [12]:
from dataclasses import dataclass
from math import isclose
from typing import Iterable, List, Sequence, Tuple

from numpy import random

## What EM is
The **Expectation-Maximization (EM)** algorithm is an iterative method for estimating model parameters when part of the data is **hidden**, **missing**, or **latent**.

A common situation is:
- you observe some data,
- but you do **not** observe the hidden cause behind each observation,
- and you still want to learn the parameters of the model.

EM alternates between:

### E-step (Expectation)
Estimate the hidden variables using the current parameter values.

In simple words:
> “Given my current guess of the model, how likely is each hidden explanation?”

### M-step (Maximization)
Update the model parameters using the expectations computed in the E-step.

In simple words:
> “Using those estimated hidden assignments, what parameter values fit the data best?”

This process repeats until the parameters stop changing much.

---

## Intuition
EM is useful when direct optimization is difficult because the full data is not completely observed.

Typical EM workflow:
1. Guess initial parameters.
2. Use them to estimate hidden-variable probabilities.
3. Re-estimate parameters from those probabilities.
4. Repeat until convergence.

So EM can be summarized as:

> **guess hidden structure -> update parameters -> repeat**

---

## When to use EM
Use EM when:
- your model contains **latent / hidden variables**,
- you can compute the expected hidden assignments from current parameters,
- and parameter updates become easy once those hidden assignments are available.

Common examples:
- **Coin problem**: unknown coin bias and unknown coin identity for each trial
- **Gaussian Mixture Models (GMMs)**: unknown cluster membership of each point
- **Hidden Markov Models (HMMs)**: unknown hidden states
- **SentencePiece unigram model**: unknown segmentation of each sentence
- **Topic models**: unknown topic assignment of words/documents

---

## EM vs K-means
K-means is closely related to EM.

- **K-means** uses **hard assignment**:
  - each point belongs to exactly one cluster
- **EM** usually uses **soft assignment**:
  - each point can belong to multiple clusters with different probabilities

So you can think of K-means as a simplified, harder version of EM for clustering.

---

## Coin example
Suppose there are two coins:
- Coin A with unknown bias `theta_A`
- Coin B with unknown bias `theta_B`

You run several experiments.
For each experiment:
1. one coin is chosen,
2. it is flipped several times,
3. you record only the results.

But you do **not** know which coin was used in each experiment.

This gives:
- **observed data**: number of heads/tails in each experiment
- **hidden variable**: which coin generated that experiment

EM solves this by:
- **E-step**: estimate the probability that each experiment came from coin A or coin B
- **M-step**: update the biases of coin A and coin B using those probabilities

---

## Advantages
- conceptually simple
- works well for latent-variable models
- often easy to implement once E-step and M-step are derived
- likelihood typically improves each iteration

## Limitations
- may converge to a **local optimum**, not the global optimum
- depends on initialization
- can be slow if the E-step or M-step is expensive
- requires a model where both steps are tractable

---

## One-sentence summary
The EM algorithm is an iterative method for learning model parameters when some important variables are hidden, by alternating between estimating the hidden structure and updating the parameters.


In [16]:
@dataclass
class Trial:
    heads: int
    tails: int

    @property
    def flips(self) -> int:
        return self.heads + self.tails


@dataclass
class EMResult:
    theta_a: float
    theta_b: float
    iterations: int
    history: List[Tuple[int, float, float]]


def likelihood(theta: float, heads: int, tails: int) -> float:
    """Bernoulli likelihood for aggregated flips, ignoring the binomial coefficient.

    For EM responsibility calculations, the shared binomial coefficient cancels out,
    so theta^heads * (1-theta)^tails is enough.
    """
    if not 0.0 < theta < 1.0:
        raise ValueError("theta must be between 0 and 1 (exclusive)")
    return (theta ** heads) * ((1.0 - theta) ** tails)


def em_two_coins(
    trials: Sequence[Trial],
    theta_a: float = 0.6,
    theta_b: float = 0.5,
    prior_a: float = 0.5,
    prior_b: float = 0.5,
    max_iter: int = 100,
    tol: float = 1e-8,
    verbose: bool = True,
) -> EMResult:
    """Run EM for the two-coin problem.

    Args:
        trials: Observed experiments as (heads, tails) counts.
        theta_a: Initial guess for coin A head probability.
        theta_b: Initial guess for coin B head probability.
        prior_a: Prior probability of choosing coin A.
        prior_b: Prior probability of choosing coin B.
        max_iter: Maximum EM iterations.
        tol: Convergence tolerance on parameter change.
        verbose: Whether to print per-iteration details.

    Returns:
        EMResult containing final parameters and history.
    """
    if not trials:
        raise ValueError("trials must not be empty")
    if not isclose(prior_a + prior_b, 1.0, rel_tol=0.0, abs_tol=1e-12):
        raise ValueError("prior_a + prior_b must equal 1")

    history: List[Tuple[int, float, float]] = []

    for iteration in range(1, max_iter + 1):
        # E-step: compute responsibilities.
        responsibilities = []
        expected_heads_a = 0.0
        expected_tails_a = 0.0
        expected_heads_b = 0.0
        expected_tails_b = 0.0

        for trial in trials:
            prob_a = likelihood(theta_a, trial.heads, trial.tails) * prior_a
            prob_b = likelihood(theta_b, trial.heads, trial.tails) * prior_b
            total = prob_a + prob_b

            if total == 0.0:
                raise ZeroDivisionError("Both probabilities are zero; try different initialization")

            # calculate P(A|T) and P(B|T)
            r_a = prob_a / total
            r_b = prob_b / total
            responsibilities.append((r_a, r_b))

            # add up expected counts for each coin
            expected_heads_a += r_a * trial.heads
            expected_tails_a += r_a * trial.tails
            expected_heads_b += r_b * trial.heads
            expected_tails_b += r_b * trial.tails

        # M-step: update parameters.
        new_theta_a = expected_heads_a / (expected_heads_a + expected_tails_a)
        new_theta_b = expected_heads_b / (expected_heads_b + expected_tails_b)
        history.append((iteration, new_theta_a, new_theta_b))

        if verbose:
            print(f"Iteration {iteration}")
            for i, trial in enumerate(trials, start=1):
                r_a, r_b = responsibilities[i - 1]
                print(
                    f"  Trial {i}: heads={trial.heads}, tails={trial.tails}, "
                    f"P(A|x)={r_a:.4f}, P(B|x)={r_b:.4f}"
                )
            print(
                f"  Expected counts A: heads={expected_heads_a:.4f}, tails={expected_tails_a:.4f}"
            )
            print(
                f"  Expected counts B: heads={expected_heads_b:.4f}, tails={expected_tails_b:.4f}"
            )
            print(f"  Updated theta_A={new_theta_a:.6f}, theta_B={new_theta_b:.6f}")
            print()

        # Check for convergence.
        if max(abs(new_theta_a - theta_a), abs(new_theta_b - theta_b)) < tol:
            theta_a, theta_b = new_theta_a, new_theta_b
            return EMResult(theta_a=theta_a, theta_b=theta_b, iterations=iteration, history=history)

        theta_a, theta_b = new_theta_a, new_theta_b

    return EMResult(theta_a=theta_a, theta_b=theta_b, iterations=max_iter, history=history)

In [21]:
def flip_coin(theta: float, count: int=10) -> Trial:
    """Simulate flipping a coin `count` times, with probability `theta`."""
    if not 0.0 < theta < 1.0:
        raise ValueError("theta must be between 0 and 1 (exclusive)")
    head = 0
    for _ in range(count):
        head += random.rand() < theta
    return Trial(heads=head, tails=count - head)

actual_a = 0.9
actual_b = 0.2
trials = []

for _ in range(20):
    trials.append(flip_coin(actual_a))
    trials.append(flip_coin(actual_b))
random.shuffle(trials)

for i in range(10):
    print(f"Trial {i+1}: {trials[i].heads} heads, {trials[i].tails} tails")

Trial 1: 0 heads, 10 tails
Trial 2: 3 heads, 7 tails
Trial 3: 3 heads, 7 tails
Trial 4: 2 heads, 8 tails
Trial 5: 3 heads, 7 tails
Trial 6: 3 heads, 7 tails
Trial 7: 9 heads, 1 tails
Trial 8: 1 heads, 9 tails
Trial 9: 0 heads, 10 tails
Trial 10: 2 heads, 8 tails


In [24]:
result = em_two_coins(
    trials,
    theta_a=0.6,
    theta_b=0.4,
    prior_a=0.5,
    prior_b=0.5,
    max_iter=50,
    tol=1e-10,
    verbose=False,
)

print("Final result")
print(f"  iterations: {result.iterations}")
print(f"  theta_A:    {result.theta_a:.6f}")
print(f"  theta_B:    {result.theta_b:.6f}")

Final result
  iterations: 11
  theta_A:    0.909204
  theta_B:    0.208716
